# TikHarm final model: train once, save the weights for the demo app

Trains the **cross-attention V+A+T** head (the best configuration in the experiments) on the creator-grouped split.
Five seeds, averaged at inference. The model sees `train + val` of the full set. **`test` is never used for training.**
It then reports the saved ensemble's score on the held-out test clips (this is the number to quote for the demo model)
and writes `final_head.pt`, which contains the weights AND the standardisation statistics the app needs.

Needs the same inputs as the three-modality fusion notebook (`emb/`, `prep/manifest_full.csv`, `text/text_features.npz`).

In [1]:
# Cell 1: load features, split, near-duplicate exclusion
import os, re, glob, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, recall_score

from sklearn.model_selection import StratifiedGroupKFold

def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def duration_match(d, bins, seed=0):
    d = d.copy(); d["dbin"] = pd.cut(d.dur, bins, labels=False, include_lowest=True); parts = []
    for (sp, b), g in d.groupby(["split", "dbin"]):
        c = g.label.value_counts()
        if len(c) < 4: continue
        n_ = int(c.min()); parts += [g[g.label == k].sample(n_, random_state=seed) for k in range(4)]
    return pd.concat(parts) if parts else d.iloc[:0]

def ensure_grouped(order, man, bins=(0, 10, 20, 40, 90, 1e9), seed=0, force=False):
    """If any creator appears in both train/val and test, apply the deterministic creator-grouped re-split
    (same code and seed as the baseline, so every notebook gets the identical split). Row order is unchanged."""
    if "path" not in man:
        print("manifest has no path column: cannot check creators"); return order
    cr = man.path.map(creator_of).values; sp = order.split.values
    if not force and not (set(cr[sp == "test"]) & set(cr[np.isin(sp, ["train", "val"])])):
        print("creator check passed: the split is already creator-grouped"); return order
    print(f"applying the creator-grouped re-split (seed {seed})" if force else "OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)")
    y = order.label.values
    rest, test = next(iter(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(order)), y, cr)))
    tr_i, va_i = next(iter(StratifiedGroupKFold(8, shuffle=True, random_state=seed + 1).split(np.zeros(len(rest)), y[rest], cr[rest])))
    new = np.array(["train"] * len(order), dtype=object); new[rest[va_i]] = "val"; new[test] = "test"
    matched = duration_match(pd.DataFrame({"uid": order.uid.values, "dur": man.dur.values, "label": y, "split": new}), list(bins), seed)
    o2 = order.copy(); o2["split"] = new; o2["in_matched"] = o2.uid.isin(matched.uid).astype(int)
    assert not (set(cr[new == "test"]) & set(cr[new != "test"])), "re-split failed"
    print(pd.crosstab(o2.cls, o2.split)); return o2

EMB = "/kaggle/input/datasets/sanawerb/emb-folder/kaggle/working/emb"
PREP = "/kaggle/input/datasets/sanawerb/preparation"
WORK = os.environ.get("WORK_DIR", "/kaggle/working")
RES = f"{WORK}/results"; os.makedirs(RES, exist_ok=True)
CFG = dict(d=256, drop=0.4, mod_drop=0.3, epochs=int(os.environ.get("EPOCHS", 40)), lr=1e-3, wd=0.05, bs=64, smooth=0.1,
           seeds=list(range(int(os.environ.get("NSEEDS", 5)))), nboot=int(os.environ.get("NBOOT", 1000)))
CLASS_NAMES = ["Safe", "Harmful", "Adult", "Suicide"]
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

order = pd.read_csv(f"{EMB}/order.csv")
files = sorted(f for f in glob.glob(f"{EMB}/chunk_*.npz") if re.search(r"chunk_\d{3}\.npz$", f))
parts = [np.load(f, allow_pickle=True) for f in files]
Fz = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
assert list(Fz["uid"]) == list(order.uid[: len(Fz["uid"])]), "chunks do not match order.csv"
N = len(Fz["uid"]); order = order.iloc[:N].copy()
man = pd.read_csv(f"{PREP}/manifest_full (1).csv").set_index("uid").loc[order.uid].reset_index()
order = ensure_grouped(order, man)

V = Fz["vit"].astype(np.float32)               # (N, 8, 768)
A = Fz["ast"].astype(np.float32)               # (N, 5, 768)
aud_ok = Fz["aud_ok"].astype(np.float32)
music_p = man["music_p"].values.astype(np.float32) if "music_p" in man else np.zeros(N, np.float32)
speech_p = man["speech_p"].values.astype(np.float32) if "speech_p" in man else np.zeros(N, np.float32)
G = np.stack([music_p, speech_p, aud_ok], 1).astype(np.float32)           # gate inputs
y = order.label.values; split = order.split.values.copy(); in_matched = order.in_matched.values.astype(bool)

# exclude test clips with a near-identical train/val neighbour (cosine > 0.95 on ViT mean embeddings)
vm = V.mean(1); vm /= np.linalg.norm(vm, axis=1, keepdims=True) + 1e-8
te_i, tr_i = np.where(split == "test")[0], np.where(split != "test")[0]
leak = te_i[(vm[te_i] @ vm[tr_i].T).max(1) > 0.95]
split[leak] = "excluded"; print("excluded near-duplicate test clips:", len(leak))


def creator_of(path):
    b = os.path.splitext(os.path.basename(path))[0]; m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def assert_creator_disjoint(paths, split_arr):
    cr = pd.Series(paths).map(creator_of).values
    shared = set(cr[split_arr == "test"]) & set(cr[np.isin(split_arr, ["train", "val"])])
    assert not shared, (f"{len(shared)} creators appear in both train/val and test. This is the OFFICIAL split, not the "
                        "creator-grouped one. Restore the grouped order.csv (re-split cell) before running.")
    print("creator check passed: no creator appears in both train/val and test")
if "path" in man: assert_creator_disjoint(man.path.values, split)

grp = np.full(N, "other audio", dtype=object)
if "music_dominant" in man: grp[man.music_dominant.values == 1] = "music-dominant"
grp[(speech_p > 0.5) & (grp != "music-dominant")] = "speech present"
print(pd.crosstab(order.cls, [split, in_matched]))
print(pd.Series(grp).value_counts().to_dict())

device: cuda
OFFICIAL split detected (creators overlap): applying the creator-grouped re-split (seed 0)
split            test  train  val
cls                              
Adult Content     218    662   68
Harmful Content   182    669  128
Safe              233    684   70
Suicide           238    669   70
excluded near-duplicate test clips: 1
creator check passed: no creator appears in both train/val and test
col_0           excluded  test       train         val      
col_1              True  False True  False True  False True 
cls                                                         
Adult Content          1   119    98   411   251    42    26
Harmful Content        0    83    99   418   251   102    26
Safe                   0   134    99   433   251    44    26
Suicide                0   139    99   418   251    44    26
{'music-dominant': 2117, 'speech present': 1292, 'other audio': 482}


In [2]:
# Cell 2: metrics, bootstrap, paired differences
def metrics(yt, P):
    pred = P.argmax(1)
    m = dict(macro_F1=f1_score(yt, pred, average="macro"),
             macro_AUC=roc_auc_score(yt, P, multi_class="ovr", average="macro", labels=[0, 1, 2, 3]),
             top2_acc=np.mean([yt[i] in np.argsort(-P[i])[:2] for i in range(len(yt))]))
    rec = recall_score(yt, pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    m.update({f"rec_{c}": r for c, r in zip(CLASS_NAMES, rec)})
    return m

def with_ci(yt, P, nboot=None):
    nboot = nboot or CFG["nboot"]; pt = metrics(yt, P); rng = np.random.default_rng(0); b = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[ix])) == 4: b.append(metrics(yt[ix], P[ix]))
    b = pd.DataFrame(b)
    return pt, {k: (np.percentile(b[k], 2.5), np.percentile(b[k], 97.5)) for k in pt}

def fmt(pt, ci, k): return f"{pt[k]:.3f} [{ci[k][0]:.3f}, {ci[k][1]:.3f}]"

def paired(yt, Pbase, Pother, nboot=None):
    nboot = nboot or CFG["nboot"]; rng = np.random.default_rng(1); d = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        d.append(f1_score(yt[ix], Pother[ix].argmax(1), average="macro") - f1_score(yt[ix], Pbase[ix].argmax(1), average="macro"))
    d = np.array(d); return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

# ---- text modality (from the text extraction notebook) ----
TEXT_DIR = "/kaggle/input/datasets/sanawerb/text-new/kaggle/working/text"
TF = np.load(f"{TEXT_DIR}/text_features.npz", allow_pickle=True)
assert list(TF["uid"]) == list(order.uid), "text_features.npz does not match order.csv (it must come from the same clip order)"
Tk = np.stack([TF["asr_emb"], TF["ocr_emb"]], 1).astype(np.float32)               # (N, 2, 384): speech, on-screen text
Mt = np.stack([TF["has_asr"], TF["has_ocr"]], 1).astype(np.float32)               # which text sources exist
Gt = np.stack([TF["has_asr"], TF["has_ocr"], np.minimum(TF["asr_words"], 50) / 50,
               np.minimum(TF["ocr_chars"], 100) / 100], 1).astype(np.float32)     # text gate inputs
tgrp = np.where(Mt.sum(1) == 2, "speech + on-screen text", np.where(Mt.sum(1) == 1, "one text source", "no text")).astype(object)
print("text availability:", pd.Series(tgrp).value_counts().to_dict())

text availability: {'one text source': 1880, 'speech + on-screen text': 1103, 'no text': 908}


In [3]:
# Cell 3: the models
class Net3(nn.Module):
    """mode: video | audio | text | early | gate | xattn.  use: which extra modalities a fusion model sees ('A', 'T' or 'AT')."""
    def __init__(s, mode, use="", d=256, p=0.4, n_cls=4):
        super().__init__(); s.mode, s.use = mode, use
        mk = lambda i: nn.Sequential(nn.LayerNorm(i), nn.Dropout(p * 0.5), nn.Linear(i, d), nn.GELU(), nn.Dropout(p))
        s.vp, s.ap, s.tp = mk(768), mk(768), mk(384)
        s.type_emb = nn.Parameter(torch.zeros(2, d))                      # tells speech tokens from on-screen-text tokens
        s.ga = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 1))
        s.gt = nn.Sequential(nn.Linear(4, 16), nn.GELU(), nn.Linear(16, 1))
        s.null = nn.Parameter(torch.zeros(1, 1, d))
        s.attn = nn.MultiheadAttention(d, 4, dropout=0.1, batch_first=True); s.an = nn.LayerNorm(d)
        k = len(use)
        din = {"video": d, "audio": d, "text": d, "early": d * (1 + k) + k, "gate": d * (1 + k), "xattn": d}[mode]
        s.head = nn.Sequential(nn.Dropout(p), nn.Linear(din, n_cls))

    def text_tokens(s, t): return s.tp(t) + s.type_emb                    # (B,2,d)
    def text_pool(s, t, mt):
        return (s.text_tokens(t) * mt[..., None]).sum(1) / mt.sum(1, keepdim=True).clamp(min=1)
    def gate_a(s, ma, ga): return torch.sigmoid(s.ga(ga)).squeeze(-1) * ma
    def gate_t(s, mt, gt): return torch.sigmoid(s.gt(gt)).squeeze(-1) * (mt.sum(1) > 0).float()

    def forward(s, v, a, ma, ga, t, mt, gt):
        if s.mode == "audio": return s.head(s.ap(a).mean(1) * ma[:, None])
        if s.mode == "text": return s.head(s.text_pool(t, mt))
        hv = s.vp(v)
        if s.mode == "video": return s.head(hv.mean(1))
        parts, anyt = [hv.mean(1)], (mt.sum(1) > 0).float()
        if s.mode == "early":
            if "A" in s.use: parts += [s.ap(a).mean(1) * ma[:, None], ma[:, None]]
            if "T" in s.use: parts += [s.text_pool(t, mt), anyt[:, None]]
            return s.head(torch.cat(parts, 1))
        if s.mode == "gate":
            if "A" in s.use: parts += [s.ap(a).mean(1) * s.gate_a(ma, ga)[:, None]]
            if "T" in s.use: parts += [s.text_pool(t, mt) * s.gate_t(mt, gt)[:, None]]
            return s.head(torch.cat(parts, 1))
        kv, keep = [s.null.expand(len(v), 1, -1)], [torch.ones(len(v), 1, dtype=torch.bool, device=v.device)]   # xattn
        if "A" in s.use: kv.append(s.ap(a)); keep.append((ma > 0)[:, None].expand(-1, a.shape[1]))
        if "T" in s.use: kv.append(s.text_tokens(t)); keep.append(mt > 0)
        kv, keep = torch.cat(kv, 1), torch.cat(keep, 1)
        att, _ = s.attn(hv, kv, kv, key_padding_mask=~keep)               # the null token is always visible, so no all-masked rows
        return s.head(s.an(hv + att).mean(1))

    @torch.no_grad()
    def gates(s, ma, ga, mt, gt): return s.gate_a(ma, ga), s.gate_t(mt, gt)

def make_data(mask):
    idx = np.where(mask)[0]; sp = split[idx]; tr = idx[sp == "train"]
    mu_v, sd_v = V[tr].reshape(-1, 768).mean(0), V[tr].reshape(-1, 768).std(0) + 1e-6
    ta = tr[aud_ok[tr] == 1]; mu_a, sd_a = A[ta].reshape(-1, 768).mean(0), A[ta].reshape(-1, 768).std(0) + 1e-6
    tt = Tk[tr][Mt[tr] > 0]; mu_t, sd_t = tt.mean(0), tt.std(0) + 1e-6                  # statistics over present text tokens only
    T = lambda x: torch.tensor(x, dtype=torch.float32, device=dev)
    Tn = ((Tk - mu_t) / sd_t) * Mt[..., None]
    D = dict(V=T((V - mu_v) / sd_v), A=T((A - mu_a) / sd_a), Ma=T(aud_ok), Ga=T(G), Tk=T(Tn), Mt=T(Mt), Gt=T(Gt),
             y=torch.tensor(y, device=dev), y_np=y)
    for s_ in ("train", "val", "test"):
        D[s_] = torch.tensor(idx[sp == s_], device=dev); D[s_ + "_np"] = idx[sp == s_]
    rng = np.random.default_rng(123); te = D["test_np"]; yt = y[te]                     # "wrong" partner: a clip of a different class
    D["wrong"] = torch.tensor([rng.choice(np.where(yt != yt[i])[0]) for i in range(len(te))], device=dev)
    return D

def batch(D, ix, ma=None, mt=None):
    return (D["V"][ix], D["A"][ix], D["Ma"][ix] if ma is None else ma, D["Ga"][ix], D["Tk"][ix],
            D["Mt"][ix] if mt is None else mt, D["Gt"][ix])

def train_one(mode, use, D, seed, mod_drop):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net3(mode, use, CFG["d"], CFG["drop"]).to(dev)
    tr = D["train"]; ytr = D["y"][tr]
    cnt = torch.bincount(ytr, minlength=4).float(); w = (len(ytr) / (4 * cnt.clamp(min=1))).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"] * int(np.ceil(len(tr) / CFG["bs"])))
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["smooth"])
    for ep in range(CFG["epochs"]):
        net.train(); perm = torch.randperm(len(tr), device=dev)
        for i in range(0, len(tr), CFG["bs"]):
            ix = tr[perm[i:i + CFG["bs"]]]; ma, mt = D["Ma"][ix], D["Mt"][ix]
            if mod_drop > 0 and mode in ("early", "gate", "xattn"):                 # hide each used modality at random
                if "A" in use: ma = ma * (torch.rand(len(ix), device=dev) >= mod_drop).float()
                if "T" in use: mt = mt * (torch.rand(len(ix), device=dev) >= mod_drop).float()[:, None]
            loss = lossf(net(*batch(D, ix, ma, mt)), D["y"][ix])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net.eval()

@torch.no_grad()
def predict(net, D, part, cond="real"):
    ix = D[part]; v, a, ma, ga, t, mt, gt = batch(D, ix)
    if cond in ("shufA", "shufT"):                                                  # take that modality from a different-class clip
        j = D["wrong"]
        if cond == "shufA": a, ma, ga = a[j], ma[j], ga[j]
        else: t, mt, gt = t[j], mt[j], gt[j]
    if cond in ("noA", "noAT"): ma = torch.zeros_like(ma)
    if cond in ("noT", "noAT"): mt = torch.zeros_like(mt)
    return torch.softmax(net(v, a, ma, ga, t, mt, gt), 1).cpu().numpy()

@torch.no_grad()
def gate_values(net, D):
    ix = D["test"]; _, _, ma, ga, _, mt, gt = batch(D, ix)
    ga_, gt_ = net.gates(ma, ga, mt, gt); return ga_.cpu().numpy(), gt_.cpu().numpy()

In [4]:
# Cell 4: smoke test (fails fast if something is wrong)
for mode, use in (("video", ""), ("audio", ""), ("text", ""), ("early", "A"), ("early", "T"), ("early", "AT"),
                  ("gate", "AT"), ("xattn", "A"), ("xattn", "T"), ("xattn", "AT")):
    net = Net3(mode, use).to(dev); B = 6
    v, a, t = torch.randn(B, 8, 768, device=dev), torch.randn(B, 5, 768, device=dev), torch.randn(B, 2, 384, device=dev)
    ga, gt = torch.rand(B, 3, device=dev), torch.rand(B, 4, device=dev)
    for ma, mt in ((1, 1), (0, 0), (1, 0), (0, 1)):                                  # every availability pattern
        out = net(v, a, torch.full((B,), float(ma), device=dev), ga, t, torch.full((B, 2), float(mt), device=dev), gt)
        assert out.shape == (B, 4) and torch.isfinite(out).all(), (mode, use, ma, mt)
        out.sum().backward()
print("all model variants run forward and backward for every audio/text availability pattern")

all model variants run forward and backward for every audio/text availability pattern


In [5]:
# Cell 5: train the final ensemble and save it
mask = np.ones(N, bool); D = make_data(mask)
tr_only = D["train_np"]                                           # statistics are computed from the train clips only, as in make_data
ta = tr_only[aud_ok[tr_only] == 1]; tt = Tk[tr_only][Mt[tr_only] > 0]
scalers = dict(mu_v=V[tr_only].reshape(-1, 768).mean(0), sd_v=V[tr_only].reshape(-1, 768).std(0) + 1e-6,
               mu_a=A[ta].reshape(-1, 768).mean(0), sd_a=A[ta].reshape(-1, 768).std(0) + 1e-6, mu_t=tt.mean(0), sd_t=tt.std(0) + 1e-6)
D["train"] = torch.cat([D["train"], D["val"]])                    # final model sees train + val; test stays held out
print(f"training on {len(D['train'])} clips, testing on {len(D['test'])} held-out clips")
t0 = time.time(); nets = []
for s_ in CFG["seeds"]:
    nets.append(train_one("xattn", "AT", D, s_, CFG["mod_drop"])); print(f"  seed {s_} done ({time.time() - t0:.0f}s)")

# held-out score of the saved ensemble, with and without audio and text
def ens(cond):
    with torch.no_grad(): return np.mean([predict(n, D, "test", cond) for n in nets], 0)
yt = y[D["test_np"]]; rows = []
for cond, label in (("real", "audio + text used"), ("noAT", "audio and text ignored")):
    P_ = ens(cond); pt, ci = with_ci(yt, P_)
    rows.append(dict(setting=label, macro_F1=fmt(pt, ci, "macro_F1"), macro_AUC=fmt(pt, ci, "macro_AUC"), **{c: round(pt[f"rec_{c}"], 2) for c in CLASS_NAMES}))
print(f"\nheld-out test (n={len(yt)}):"); print(pd.DataFrame(rows).to_string(index=False))

ck = dict(mode="xattn", use="AT", cfg=dict(d=CFG["d"], drop=CFG["drop"]), class_names=CLASS_NAMES,
          states=[{k: v.cpu() for k, v in n.state_dict().items()} for n in nets],
          scalers={k: torch.tensor(np.asarray(v), dtype=torch.float32) for k, v in scalers.items()})
os.makedirs(f"{WORK}/artifacts", exist_ok=True)
torch.save(ck, f"{WORK}/artifacts/final_head.pt")
json.dump(dict(trained_on="train+val of the full clean set, creator-grouped split (seed 0)", seeds=CFG["seeds"], test_n=int(len(yt)),
               test_results=rows, created=time.strftime("%Y-%m-%d %H:%M")), open(f"{WORK}/artifacts/final_head_card.json", "w"), indent=2)
print(f"saved {WORK}/artifacts/final_head.pt ({os.path.getsize(f'{WORK}/artifacts/final_head.pt') / 1e6:.1f} MB) and final_head_card.json")

training on 3020 clips, testing on 870 held-out clips
  seed 0 done (13s)
  seed 1 done (24s)
  seed 2 done (34s)
  seed 3 done (45s)
  seed 4 done (55s)

held-out test (n=870):
               setting             macro_F1            macro_AUC  Safe  Harmful  Adult  Suicide
     audio + text used 0.882 [0.860, 0.902] 0.977 [0.969, 0.984]  0.96     0.77   0.83     0.96
audio and text ignored 0.870 [0.846, 0.892] 0.969 [0.961, 0.978]  0.92     0.77   0.84     0.94
saved /kaggle/working/artifacts/final_head.pt (15.3 MB) and final_head_card.json


In [6]:
# Cell 6: check that the saved file loads and reproduces the same predictions
ck2 = torch.load(f"{WORK}/artifacts/final_head.pt", map_location=dev, weights_only=True)
nets2 = []
for st in ck2["states"]:
    n_ = Net3(ck2["mode"], ck2["use"], ck2["cfg"]["d"], ck2["cfg"]["drop"]).to(dev); n_.load_state_dict(st); nets2.append(n_.eval())
with torch.no_grad(): P2 = np.mean([predict(n, D, "test", "real") for n in nets2], 0)
print("reload check, max difference in test probabilities:", float(np.abs(P2 - ens("real")).max()))
assert np.abs(P2 - ens("real")).max() < 1e-4

reload check, max difference in test probabilities: 0.0


In [7]:
!zip -r artifacts.zip /kaggle/working/artifacts

  adding: kaggle/working/artifacts/ (stored 0%)
  adding: kaggle/working/artifacts/final_head.pt (deflated 8%)
  adding: kaggle/working/artifacts/final_head_card.json (deflated 55%)
